# SCBE Code A/B Matched-Budget Benchmark (v2)

Matched-budget code training comparison: baseline L3-only vs triangulated KO/CA/DR Sacred Tongue encoding.

**Key fix from v1:** 150 steps (not 75) to get past the crossover point where triangulated training overtakes baseline. Full loss history saved + results pushed to HuggingFace so nothing gets lost.

| Setting | Value |
|---------|-------|
| Model | `Qwen/Qwen2.5-Coder-0.5B-Instruct` |
| LoRA | r=8, alpha=16, targets q/k/v/o_proj |
| Steps | 150 per condition |
| Batch | 2 × 8 gradient accum = effective 16 |
| Budget | Matched token count between conditions |
| Runtime | ~30-45 min on free Colab T4 |

In [ ]:
# Install dependencies
!pip install -q transformers datasets accelerate peft trl bitsandbytes huggingface_hub pytest


In [ ]:
# Clone the repo (private — needs HF or GitHub token)
import os, subprocess
from google.colab import userdata

REPO_DIR = "/content/SCBE-AETHERMOORE"
if not os.path.exists(REPO_DIR):
    # Try using a GitHub PAT from Colab secrets
    try:
        gh_token = userdata.get("GITHUB_TOKEN")
        clone_url = f"https://{gh_token}@github.com/issdandavis/SCBE-AETHERMOORE.git"
    except Exception:
        # Fall back to SSH or public clone attempt
        clone_url = "https://github.com/issdandavis/SCBE-AETHERMOORE.git"
    subprocess.run(["git", "clone", "--depth", "1", clone_url, REPO_DIR], check=True)
else:
    print("Repo already present")

%cd /content/SCBE-AETHERMOORE

In [ ]:
# Optional: login to Hugging Face if you want to push results later
from huggingface_hub import login
login()


In [ ]:
# Build matched-budget corpora locally
!python scripts/research/train_code_ab_fast.py --prepare-only


In [ ]:
# Inspect the matched-budget manifest
import json
from pathlib import Path

manifest_path = Path("artifacts/research/code_ab_fast/manifest.json")
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
print(json.dumps(manifest, indent=2))


In [ ]:
# Matched-budget A/B training on GPU (150 steps for crossover)
import json
import time
from datetime import datetime, timezone
from pathlib import Path

import torch
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, Trainer

BASE_MODEL = "Qwen/Qwen2.5-Coder-0.5B-Instruct"
MAX_STEPS = 150  # v1 used 75; need 150+ for crossover effect
TIMESTAMP = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN_ROOT = Path(f"/content/code_ab_{TIMESTAMP}")
RUN_ROOT.mkdir(parents=True, exist_ok=True)

def train_condition(data_path: str, run_name: str):
    print(f"\n{'='*60}")
    print(f"  Training: {run_name}")
    print(f"  Data: {data_path}")
    print(f"  Max steps: {MAX_STEPS}")
    print(f"{'='*60}\n")

    ds = load_dataset("json", data_files=data_path, split="train")
    print(f"Loaded {len(ds)} rows")

    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )
    model = prepare_model_for_kbit_training(model)
    lora = LoraConfig(
        r=8,
        lora_alpha=16,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    )
    model = get_peft_model(model, lora)

    def tokenize_fn(batch):
        return tokenizer(batch["text"], truncation=True, max_length=512, padding="max_length")

    tokenized = ds.map(tokenize_fn, batched=True, remove_columns=["text"])
    tokenized = tokenized.map(lambda batch: {"labels": batch["input_ids"]}, batched=True)

    args = TrainingArguments(
        output_dir=str(RUN_ROOT / run_name),
        num_train_epochs=1,
        max_steps=MAX_STEPS,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        learning_rate=2e-4,
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        logging_steps=5,  # Log every 5 steps for detailed curve
        save_strategy="no",
        bf16=torch.cuda.is_available(),
        gradient_checkpointing=True,
        report_to="none",
    )

    trainer = Trainer(model=model, args=args, train_dataset=tokenized)
    t0 = time.time()
    trainer.train()
    elapsed = time.time() - t0

    # Capture full loss history for crossover analysis
    loss_history = []
    for entry in trainer.state.log_history:
        if "loss" in entry and "step" in entry:
            loss_history.append({"step": entry["step"], "loss": round(float(entry["loss"]), 4)})

    final_loss = loss_history[-1]["loss"] if loss_history else None

    result = {
        "run_name": run_name,
        "model": BASE_MODEL,
        "max_steps": MAX_STEPS,
        "rows": len(ds),
        "elapsed_seconds": round(elapsed, 2),
        "final_loss": final_loss,
        "loss_history": loss_history,
        "device": str(next(model.parameters()).device),
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
    }
    del model, trainer
    torch.cuda.empty_cache()
    return result

# Run both conditions
baseline_path = "artifacts/research/code_ab_fast/baseline_matched.jsonl"
triangulated_path = "artifacts/research/code_ab_fast/triangulated_matched.jsonl"

baseline_result = train_condition(baseline_path, "baseline_matched")
triangulated_result = train_condition(triangulated_path, "triangulated_matched")

# Build summary with crossover analysis
summary = {
    "experiment": "SCBE Code A/B Matched-Budget (v2)",
    "timestamp": TIMESTAMP,
    "baseline": baseline_result,
    "triangulated": triangulated_result,
}

b_loss = baseline_result["final_loss"]
t_loss = triangulated_result["final_loss"]
if b_loss is not None and t_loss is not None:
    delta = t_loss - b_loss
    summary["delta_loss"] = round(delta, 4)
    summary["relative_improvement_pct"] = round(abs(delta) / b_loss * 100, 2)
    summary["winner"] = "triangulated" if t_loss < b_loss else "baseline"

    # Find crossover step (where triangulated first beats baseline)
    b_hist = {e["step"]: e["loss"] for e in baseline_result["loss_history"]}
    t_hist = {e["step"]: e["loss"] for e in triangulated_result["loss_history"]}
    crossover_step = None
    for step in sorted(set(b_hist.keys()) & set(t_hist.keys())):
        if t_hist[step] < b_hist[step]:
            crossover_step = step
            break
    summary["crossover_step"] = crossover_step

# Save locally
summary_path = RUN_ROOT / "code_ab_summary.json"
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
print("\n" + "="*60)
print("  RESULTS")
print("="*60)
print(json.dumps({k: v for k, v in summary.items() if k not in ("baseline", "triangulated")}, indent=2))
print(f"\nBaseline final loss:     {b_loss}")
print(f"Triangulated final loss: {t_loss}")
if summary.get("crossover_step"):
    print(f"Crossover at step:       {summary['crossover_step']}")
print(f"\nFull results saved to: {summary_path}")

In [ ]:
# Push results to HuggingFace so they survive Colab session death
from huggingface_hub import HfApi

api = HfApi()
repo_id = "issdandavis/scbe-aethermoore-training-data"

# Upload the summary JSON
api.upload_file(
    path_or_fileobj=str(summary_path),
    path_in_repo=f"experiments/code_ab_v2_{TIMESTAMP}.json",
    repo_id=repo_id,
    repo_type="dataset",
    commit_message=f"Code A/B v2 results ({TIMESTAMP}): {summary.get('relative_improvement_pct', '?')}% delta",
)
print(f"Results pushed to https://huggingface.co/datasets/{repo_id}")
print(f"Path: experiments/code_ab_v2_{TIMESTAMP}.json")
print("Results are now safe even if Colab session dies.")